In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- NEW BLOCK: INSTALL GNN LIBRARIES ---
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.6 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu118.html
ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib


In [3]:
!pip install gdown


In [4]:
!gdown 1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ

# https://drive.google.com/file/d/1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ/view?usp=drive_link

Downloading...
From (original): https://drive.google.com/uc?id=1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ
From (redirected): https://drive.google.com/uc?id=1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ&confirm=t&uuid=bf13bc0a-3f14-496e-8920-5b85f4f78b90
To: /content/suitesparse_kaggle_export.zip
100% 1.72G/1.72G [00:28<00:00, 60.2MB/s]


In [5]:
!unzip suitesparse_kaggle_export.zip


Streaming output truncated to the last 5000 lines.
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1642.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1643.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1644.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1645.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1646.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1647.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1648.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1649.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_165.txt  
  inflating: suitespars

In [6]:
!pip install ssgetpy -q


In [7]:
import json
import ssgetpy
from pathlib import Path

data_dir = Path("suitesparse_mtx")

# Load the saved metadata
with open("suitesparse_selected.json") as f:
    meta = json.load(f)

print("Matrices listed in JSON:", len(meta))

# Rebuild selected list exactly as in kaggle
selected = []
for item in meta:
    g = item["group"]
    n = item["name"]
    try:
        # Fetch the exact matrix entry from SuiteSparse DB metadata
        m = ssgetpy.search(group=g, name=n, limit=1)[0]
        selected.append(m)
    except:
        print("Could not find:", g, n)

print("Reconstructed selected list:", len(selected))


Matrices listed in JSON: 1765
Reconstructed selected list: 1765


In [ ]:
from pathlib import Path
import scipy.io
import scipy.sparse as sp
import numpy as np

data_dir = Path("suitesparse_mtx")

def load_matrix_metadata(m):
    """
    Given an ssgetpy Matrix object m (with .group and .name),
    recursively find a .mtx file under suitesparse_mtx/group/name/,
    load it as SciPy CSR (float64), and return it.
    """
    base_dir = data_dir / m.group / m.name

    if not base_dir.exists():
        print(f"[WARN] Directory not found for {m.group}/{m.name}: {base_dir}")
        return None

    # Recursively search for any .mtx file
    mtx_file = None
    for f in base_dir.rglob("*.mtx"):
        mtx_file = f
        break  # first match

    if mtx_file is None:
        print(f"[WARN] No .mtx file found under {base_dir}")
        return None

    print(f"Loading {m.group}/{m.name} from {mtx_file.relative_to(data_dir)}")
    A = scipy.io.mmread(str(mtx_file))
    A = A.tocsr().astype(np.float64)   # ensure CSR float64

    print(f"  shape: {A.shape}, nnz: {A.nnz}, dtype: {A.dtype}")
    return A

#  Test on first 3 matrices
print("Testing loader on a few matrices...")
for m in selected[:3]:
    A = load_matrix_metadata(m)
    print("-" * 60)


Testing loader on a few matrices...
Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
------------------------------------------------------------
Loading HB/494_bus from HB/494_bus/494_bus/494_bus.mtx
  shape: (494, 494), nnz: 1666, dtype: float64
------------------------------------------------------------
Loading HB/662_bus from HB/662_bus/662_bus/662_bus.mtx
  shape: (662, 662), nnz: 2474, dtype: float64
------------------------------------------------------------


In [9]:
# Install CuPy for CUDA 12 (works on current Colab GPUs)
!pip install -q cupy-cuda12x

import cupy as cp
import time
import scipy.sparse as sp
import numpy as np

def scipy_to_cupy_csr(A, dtype=cp.float64):
    """
    Convert a SciPy CSR matrix A to a CuPy CSR matrix with given dtype.
    """
    A = A.tocsr()
    data   = cp.asarray(A.data,   dtype=dtype)
    indices = cp.asarray(A.indices, dtype=cp.int32)
    indptr  = cp.asarray(A.indptr,  dtype=cp.int32)
    return cp.sparse.csr_matrix((data, indices, indptr), shape=A.shape)

def gpu_reference_spmv(A, n_runs=5):
    """
    Reference SpMV on GPU in float64 precision.

    Returns:
        A64_gpu : CuPy CSR (float64)
        x_ref   : CuPy dense vector (float64)
        y_ref   : CuPy dense vector (float64)
        avg_t   : average execution time over n_runs (seconds)
    """
    # 1) Move A to GPU in float64
    A64_gpu = scipy_to_cupy_csr(A, dtype=cp.float64)

    n = A.shape[1]
    x_ref = cp.random.randn(n, dtype=cp.float64)

    # 2) Warm-up (to avoid including one-time overhead)
    _ = A64_gpu @ x_ref
    cp.cuda.Stream.null.synchronize()

    # 3) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_ref = A64_gpu @ x_ref
    cp.cuda.Stream.null.synchronize()

    avg_t = (time.time() - t0) / n_runs
    return A64_gpu, x_ref, y_ref, avg_t

# Quick sanity test on the first matrix
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Test matrix: {m_test.group}/{m_test.name}")
print("  shape:", A_test.shape, "nnz:", A_test.nnz)
print(f"  Reference GPU SpMV avg time: {t_ref:.6e} s")
print("  ||y_ref|| (L2 norm):", float(cp.linalg.norm(y_ref)))


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Test matrix: HB/1138_bus
  shape: (1138, 1138) nnz: 4054
  Reference GPU SpMV avg time: 1.656055e-04 s
  ||y_ref|| (L2 norm): 121370.44415805786


In [ ]:
import scipy.sparse as sp

def build_entrywise_matrices(A, thresh=1.0):
    """
    Split entries of A by magnitude:

      - A_hi: |a_ij| >= thresh
      - A_lo: |a_ij| <  thresh

    Returns two SciPy CSR matrices.
    """
    A = A.tocoo()
    data = A.data
    rows = A.row
    cols = A.col

    mask_hi = np.abs(data) >= thresh
    mask_lo = ~mask_hi

    A_hi = sp.coo_matrix((data[mask_hi], (rows[mask_hi], cols[mask_hi])),
                         shape=A.shape).tocsr()
    A_lo = sp.coo_matrix((data[mask_lo], (rows[mask_lo], cols[mask_lo])),
                         shape=A.shape).tocsr()
    return A_hi, A_lo

def eval_entrywise_spmv(A, x_ref, y_ref, thresh=1.0, n_runs=5):
    """
    Entry-wise mixed precision SpMV:

      - A_hi stored and multiplied in float64
      - A_lo stored and multiplied in float32
      - Both results accumulated in float64

    A: SciPy sparse matrix (CSR/COO/etc)
    x_ref, y_ref: CuPy vectors from gpu_reference_spmv
    """
    # 1) Split into high/low magnitude matrices on CPU
    A_hi, A_lo = build_entrywise_matrices(A, thresh=thresh)

    # 2) Move to GPU with desired precisions
    A_hi_gpu = scipy_to_cupy_csr(A_hi, dtype=cp.float64)
    A_lo_gpu = scipy_to_cupy_csr(A_lo, dtype=cp.float32)

    # 3) Prepare x in both precisions
    x64 = x_ref
    x32 = x_ref.astype(cp.float32)

    # 4) Warm-up
    _ = A_hi_gpu @ x64
    _ = A_lo_gpu @ x32
    cp.cuda.Stream.null.synchronize()

    # 5) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_hi = A_hi_gpu @ x64
        y_lo = A_lo_gpu @ x32
        y    = y_hi + y_lo.astype(cp.float64)
    cp.cuda.Stream.null.synchronize()
    avg_t = (time.time() - t0) / n_runs

    # 6) Relative error vs reference
    rel_err = cp.linalg.norm(y - y_ref) / cp.linalg.norm(y_ref)
    return avg_t, float(rel_err)

# Quick test on the same test matrix as before
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Reference time (float64): {t_ref:.6e} s")

t_entry, e_entry = eval_entrywise_spmv(A_test, x_ref, y_ref, thresh=1.0)
print(f"Entry-wise time: {t_entry:.6e} s,  rel_err = {e_entry:.3e}")


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Reference time (float64): 1.291752e-04 s
Entry-wise time: 8.026428e-02 s,  rel_err = 1.758e-12


In [ ]:
def build_rowwise_matrices(A, frac_high=0.3):
    """
    Split rows by importance using max |a_ij| per row.

    Top frac_high fraction of rows go to high precision (A_hi),
    remaining rows go to low precision (A_lo).
    """
    A = A.tocsr()
    n_rows = A.shape[0]
    row_max = np.zeros(n_rows, dtype=np.float64)

    data = A.data
    indptr = A.indptr

    # Compute max |a_ij| per row
    for i in range(n_rows):
        s, e = indptr[i], indptr[i+1]
        if s == e:
            row_max[i] = 0.0
        else:
            row_max[i] = np.max(np.abs(data[s:e]))

    # Determine threshold to pick top frac_high rows
    q = np.quantile(row_max, 1.0 - frac_high)
    hi_mask = row_max >= q
    lo_mask = ~hi_mask

    A_hi = A[hi_mask, :]
    A_lo = A[lo_mask, :]

    return A_hi, A_lo, hi_mask, lo_mask

def eval_rowwise_spmv(A, x_ref, y_ref, frac_high=0.3, n_runs=5):
    """
    Row-wise mixed precision SpMV:

      - High-importance rows in float64 (A_hi)
      - Remaining rows in float32 (A_lo)

    A: SciPy sparse matrix
    x_ref, y_ref: CuPy vectors from gpu_reference_spmv
    """
    # 1) Build high/low row blocks
    A_hi, A_lo, hi_mask, lo_mask = build_rowwise_matrices(A, frac_high=frac_high)

    # 2) Move row blocks to GPU
    A_hi_gpu = scipy_to_cupy_csr(A_hi, dtype=cp.float64)
    A_lo_gpu = scipy_to_cupy_csr(A_lo, dtype=cp.float32)

    x64 = x_ref
    x32 = x_ref.astype(cp.float32)

    # 3) Warm-up
    _ = A_hi_gpu @ x64
    _ = A_lo_gpu @ x32
    cp.cuda.Stream.null.synchronize()

    # 4) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_hi = A_hi_gpu @ x64   # high-precision rows
        y_lo = A_lo_gpu @ x32   # low-precision rows
    cp.cuda.Stream.null.synchronize()
    avg_t = (time.time() - t0) / n_runs

    # 5) Reconstruct full y in original row order
    y_full = cp.zeros_like(y_ref)
    hi_idx = np.where(hi_mask)[0]
    lo_idx = np.where(lo_mask)[0]

    y_full[hi_idx] = y_hi
    y_full[lo_idx] = y_lo.astype(cp.float64)

    # 6) Relative error vs reference
    rel_err = cp.linalg.norm(y_full - y_ref) / cp.linalg.norm(y_ref)
    return avg_t, float(rel_err)

#  Quick test on the same test matrix
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Reference time (float64): {t_ref:.6e} s")

t_row, e_row = eval_rowwise_spmv(A_test, x_ref, y_ref, frac_high=0.3)
print(f"Row-wise time: {t_row:.6e} s,  rel_err = {e_row:.3e}")


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Reference time (float64): 7.972717e-05 s
Row-wise time: 2.681732e-04 s,  rel_err = 1.132e-09


In [12]:
def build_adaptive_matrices(A, quantile=0.7):
    """
    2-bucket adaptive split by |a_ij|:
      - A_hi: entries >= quantile(|a|)
      - A_lo: entries <  quantile(|a|)
    """
    A = A.tocoo()
    data = A.data
    rows = A.row
    cols = A.col

    abs_data = np.abs(data)
    if abs_data.size == 0:
        Z = sp.csr_matrix(A.shape, dtype=np.float64)
        return Z, Z

    thresh = np.quantile(abs_data, quantile)

    mask_hi = abs_data >= thresh
    mask_lo = abs_data <  thresh

    A_hi = sp.coo_matrix((data[mask_hi], (rows[mask_hi], cols[mask_hi])),
                         shape=A.shape).tocsr()
    A_lo = sp.coo_matrix((data[mask_lo], (rows[mask_lo], cols[mask_lo])),
                         shape=A.shape).tocsr()

    return A_hi, A_lo

def eval_adaptive_spmv(A, x_ref, y_ref, quantile=0.7, n_runs=5):
    """
    Adaptive mixed precision SpMV:

      - High bucket (>= quantile) -> float64
      - Low bucket  (< quantile)  -> float32
    """
    # 1) Build high/low matrices
    A_hi, A_lo = build_adaptive_matrices(A, quantile=quantile)

    # 2) Move to GPU with desired precision
    A_hi_gpu = scipy_to_cupy_csr(A_hi, dtype=cp.float64)
    A_lo_gpu = scipy_to_cupy_csr(A_lo, dtype=cp.float32)

    x64 = x_ref
    x32 = x_ref.astype(cp.float32)

    # 3) Warm-up
    _ = A_hi_gpu @ x64
    _ = A_lo_gpu @ x32
    cp.cuda.Stream.null.synchronize()

    # 4) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_hi = A_hi_gpu @ x64
        y_lo = A_lo_gpu @ x32
        y    = y_hi + y_lo.astype(cp.float64)
    cp.cuda.Stream.null.synchronize()

    avg_t  = (time.time() - t0) / n_runs
    rel_err = cp.linalg.norm(y - y_ref) / cp.linalg.norm(y_ref)
    return avg_t, float(rel_err)

# Quick test on the same test matrix
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Reference time (float64): {t_ref:.6e} s")

t_adapt, e_adapt = eval_adaptive_spmv(A_test, x_ref, y_ref, quantile=0.7)
print(f"Adaptive time: {t_adapt:.6e} s,  rel_err = {e_adapt:.3e}")


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Reference time (float64): 1.805782e-04 s
Adaptive time: 4.303455e-04 s,  rel_err = 6.701e-10


In [ ]:
# ===== Simple, robust labeling loop using your existing loader =====

import json
from tqdm.auto import tqdm

err_tol = 1e-3   # maximum allowed relative error
labels = {}      # "group/name" -> 0/1/2
stats  = {}      # detailed timings + errors 

skipped_error = []   # matrices that failed for any reason

for m in tqdm(selected, desc="Labeling matrices"):
    name = f"{m.group}/{m.name}"

    try:
        # 1) Load matrix using your existing helper
        A = load_matrix_metadata(m)
        if A is None:
            print(f"[SKIP] Could not load {name}")
            skipped_error.append(name)
            continue

        # 2) Reference SpMV (float64 on GPU)
        A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A)

        # 3) Evaluate three strategies
        t_ent, e_ent = eval_entrywise_spmv(A, x_ref, y_ref, thresh=1.0)
        t_row, e_row = eval_rowwise_spmv(A, x_ref, y_ref, frac_high=0.3)
        t_adp, e_adp = eval_adaptive_spmv(A, x_ref, y_ref, quantile=0.7)

        methods = [
            {"name": "entrywise", "time": t_ent, "err": e_ent, "label": 0},
            {"name": "rowwise",   "time": t_row, "err": e_row, "label": 1},
            {"name": "adaptive",  "time": t_adp, "err": e_adp, "label": 2},
        ]

        # 4) Save raw stats
        stats[name] = methods

        # 5) Filter by error tolerance
        feasible = [mt for mt in methods if mt["err"] <= err_tol]

        if feasible:
            # Among feasible ones, pick the fastest
            best = min(feasible, key=lambda d: d["time"])
        else:
            # If nothing meets error tolerance, pick the one with smallest error
            best = min(methods, key=lambda d: d["err"])

        # 6) Store best label
        labels[name] = best["label"]

    except Exception as e:
        # This will catch things like *_label.mtx becoming numpy.ndarray inside load_matrix_metadata
        print(f"[ERROR] {name}: {e}")
        skipped_error.append(name)
        continue

# 7) Save results
with open("labels.json", "w") as f:
    json.dump(labels, f, indent=2)

with open("stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("Done. Labeled matrices:", len(labels))
print("Label distribution:", {
    lbl: list(labels.values()).count(lbl) for lbl in sorted(set(labels.values()))
})

print("\nSkipped due to errors (including label/metadata matrices):", len(skipped_error))


Labeling matrices:   0%|          | 0/1765 [00:00<?, ?it/s]

Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Loading HB/494_bus from HB/494_bus/494_bus/494_bus.mtx
  shape: (494, 494), nnz: 1666, dtype: float64
Loading HB/662_bus from HB/662_bus/662_bus/662_bus.mtx
  shape: (662, 662), nnz: 2474, dtype: float64
Loading HB/685_bus from HB/685_bus/685_bus/685_bus.mtx
  shape: (685, 685), nnz: 3249, dtype: float64
Loading HB/arc130 from HB/arc130/arc130/arc130.mtx
  shape: (130, 130), nnz: 1282, dtype: float64
Loading HB/bcsstk01 from HB/bcsstk01/bcsstk01/bcsstk01.mtx
  shape: (48, 48), nnz: 400, dtype: float64
Loading HB/bcsstk02 from HB/bcsstk02/bcsstk02/bcsstk02.mtx
  shape: (66, 66), nnz: 4356, dtype: float64
Loading HB/bcsstk03 from HB/bcsstk03/bcsstk03/bcsstk03.mtx
  shape: (112, 112), nnz: 640, dtype: float64
Loading HB/bcsstk04 from HB/bcsstk04/bcsstk04/bcsstk04.mtx
  shape: (132, 132), nnz: 3648, dtype: float64
Loading HB/bcsstk05 from HB/bcsstk05/bcsstk05/bcsstk05.mtx
  shape: (1

In [ ]:
import numpy as np

def sparse_to_image(A, H=128, W=128):
    """
    Helper function to convert sparse matrix to a fixed grid.
    Used by matrix_to_graph to determine node connectivity.
    """
    A = A.tocoo()
    n_rows, n_cols = A.shape

    if A.nnz == 0:
        return np.zeros((H, W), dtype=np.float32)

    rows = A.row.astype(np.float64)
    cols = A.col.astype(np.float64)
    vals = A.data.astype(np.float64)

    # 1) Scale coordinates to [0, H-1] and [0, W-1]
    r_scaled = np.floor(rows * (H - 1) / max(n_rows - 1, 1)).astype(int)
    c_scaled = np.floor(cols * (W - 1) / max(n_cols - 1, 1)).astype(int)

    # 2) Normalize |vals| to [0,1]
    v_abs = np.abs(vals)
    v_min, v_max = v_abs.min(), v_abs.max()
    if v_max - v_min < 1e-12:
        v_norm = np.zeros_like(v_abs)
    else:
        v_norm = (v_abs - v_min) / (v_max - v_min)

    # 3) Accumulate into image
    img = np.zeros((H, W), dtype=np.float32)
    np.add.at(img, (r_scaled, c_scaled), v_norm)

    # 4) Normalize whole image to [0,1]
    max_val = img.max()
    if max_val > 0:
        img /= max_val

    return img

In [17]:
import torch
import numpy as np
import json
from tqdm import tqdm
from torch_geometric.data import Data
from scipy.sparse import coo_matrix

# --- 1. Load Labels ---
with open("labels.json", "r") as f:
    labels = json.load(f)

# --- 2. Define Conversion Function (Matrix -> Graph) ---
def matrix_to_graph(A_sparse, label, H=128, W=128):
    # Step A: use sparse_to_image logic to get a fixed 128x128 grid
    img = sparse_to_image(A_sparse, H=H, W=W)

    # Step B: Convert image back to sparse coordinates
    A_coo = coo_matrix(img)
    row_indices = A_coo.row
    col_indices = A_coo.col

    # Step C: Build Graph Edges (Bipartite: Rows 0..127, Cols 128..255)
    source_nodes = torch.tensor(row_indices, dtype=torch.long)
    target_nodes = torch.tensor(col_indices + H, dtype=torch.long)
    edge_index = torch.stack([source_nodes, target_nodes], dim=0)

    # Make edges undirected
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

    # Step D: Node Features (Degree)
    num_nodes = H + W
    degrees = torch.zeros(num_nodes, 1, dtype=torch.float)

    r_unique, r_counts = np.unique(row_indices, return_counts=True)
    degrees[r_unique] = torch.tensor(r_counts, dtype=torch.float).unsqueeze(1)

    c_unique, c_counts = np.unique(col_indices, return_counts=True)
    degrees[c_unique + H] = torch.tensor(c_counts, dtype=torch.float).unsqueeze(1)

    degrees = torch.log1p(degrees)
    y_tensor = torch.tensor([label], dtype=torch.long)

    return Data(x=degrees, edge_index=edge_index, y=y_tensor)

# --- 3. Build Dataset with Augmentation ---
graph_list = []
N_AUG = 10
minority_classes = [0, 2]

print(f"Building Graphs...")

for m in tqdm(selected, desc="Processing"):
    name = f"{m.group}/{m.name}"
    if name not in labels:
        continue

    A = load_matrix_metadata(m)
    if A is None:
        continue

    label = labels[name]

    try:
        # Create Base Graph
        base_graph = matrix_to_graph(A, label)
        graph_list.append(base_graph)

        # Apply Augmentation for Minority Classes
        if label in minority_classes:
            for _ in range(N_AUG):
                aug_graph = matrix_to_graph(A, label)
                num_edges = aug_graph.edge_index.shape[1]
                if num_edges > 0:
                    mask = torch.rand(num_edges) > 0.05
                    aug_graph.edge_index = aug_graph.edge_index[:, mask]
                graph_list.append(aug_graph)
    except Exception as e:
        print(f"Error processing {name}: {e}")
        continue

print(f"\nFinal Graph Dataset Size: {len(graph_list)}")

Building Graphs...


Processing:   1%|          | 11/1765 [00:00<00:16, 108.15it/s]

Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Loading HB/494_bus from HB/494_bus/494_bus/494_bus.mtx
  shape: (494, 494), nnz: 1666, dtype: float64
Loading HB/662_bus from HB/662_bus/662_bus/662_bus.mtx
  shape: (662, 662), nnz: 2474, dtype: float64
Loading HB/685_bus from HB/685_bus/685_bus/685_bus.mtx
  shape: (685, 685), nnz: 3249, dtype: float64
Loading HB/arc130 from HB/arc130/arc130/arc130.mtx
  shape: (130, 130), nnz: 1282, dtype: float64
Loading HB/bcsstk01 from HB/bcsstk01/bcsstk01/bcsstk01.mtx
  shape: (48, 48), nnz: 400, dtype: float64
Loading HB/bcsstk02 from HB/bcsstk02/bcsstk02/bcsstk02.mtx
  shape: (66, 66), nnz: 4356, dtype: float64
Loading HB/bcsstk03 from HB/bcsstk03/bcsstk03/bcsstk03.mtx
  shape: (112, 112), nnz: 640, dtype: float64
Loading HB/bcsstk04 from HB/bcsstk04/bcsstk04/bcsstk04.mtx
  shape: (132, 132), nnz: 3648, dtype: float64
Loading HB/bcsstk05 from HB/bcsstk05/bcsstk05/bcsstk05.mtx
  shape: (1

Processing:   1%|          | 22/1765 [00:00<00:44, 39.53it/s] 

Loading HB/bcsstk18 from HB/bcsstk18/bcsstk18/bcsstk18.mtx
  shape: (11948, 11948), nnz: 149090, dtype: float64
Loading HB/bcsstk19 from HB/bcsstk19/bcsstk19/bcsstk19.mtx
  shape: (817, 817), nnz: 6853, dtype: float64
Loading HB/bcsstk20 from HB/bcsstk20/bcsstk20/bcsstk20.mtx
  shape: (485, 485), nnz: 3135, dtype: float64
Loading HB/bcsstk21 from HB/bcsstk21/bcsstk21/bcsstk21.mtx
  shape: (3600, 3600), nnz: 26600, dtype: float64
Loading HB/bcsstk22 from HB/bcsstk22/bcsstk22/bcsstk22.mtx
  shape: (138, 138), nnz: 696, dtype: float64
Loading HB/bcsstk23 from HB/bcsstk23/bcsstk23/bcsstk23.mtx
  shape: (3134, 3134), nnz: 45178, dtype: float64
Loading HB/bcsstk24 from HB/bcsstk24/bcsstk24/bcsstk24.mtx


...

  shape: (38547, 38547), nnz: 618158, dtype: float64
Loading ML_Graph/optdigits_10NN from ML_Graph/optdigits_10NN/optdigits_10NN/optdigits_10NN.mtx
  shape: (5620, 5620), nnz: 79650, dtype: float64


Processing:  99%|█████████▉| 1756/1765 [00:54<00:00, 29.78it/s]

Loading ML_Graph/Plants_10NN from ML_Graph/Plants_10NN/Plants_10NN/Plants_10NN.mtx
  shape: (1600, 1600), nnz: 21930, dtype: float64
Loading ML_Graph/plantsmargin_12NN from ML_Graph/plantsmargin_12NN/plantsmargin_12NN/plantsmargin_12NN.mtx
  shape: (1600, 1600), nnz: 25482, dtype: float64
Loading ML_Graph/plantstexture_10NN from ML_Graph/plantstexture_10NN/plantstexture_10NN/plantstexture_10NN.mtx
  shape: (1599, 1599), nnz: 21204, dtype: float64
Loading ML_Graph/Spectro_10NN from ML_Graph/Spectro_10NN/Spectro_10NN/Spectro_10NN.mtx
  shape: (531, 531), nnz: 7422, dtype: float64
Loading ML_Graph/usps_norm_5NN from ML_Graph/usps_norm_5NN/usps_norm_5NN/usps_norm_5NN.mtx
  shape: (11000, 11000), nnz: 81112, dtype: float64
Loading ML_Graph/Vehicle_10NN from ML_Graph/Vehicle_10NN/Vehicle_10NN/Vehicle_10NN.mtx
  shape: (846, 846), nnz: 10894, dtype: float64
Loading FlowIPM22/uni_chimera_i2 from FlowIPM22/uni_chimera_i2/uni_chimera_i2/uni_chimera_i2_A_21.mtx
  shape: (100000, 100000), nnz: 797

Processing: 100%|██████████| 1765/1765 [00:56<00:00, 31.08it/s]

Loading FlowIPM22/uni_chimera_i5 from FlowIPM22/uni_chimera_i5/uni_chimera_i5/uni_chimera_i5_A_16.mtx
  shape: (100000, 100000), nnz: 499982, dtype: float64

Final Graph Dataset Size: 2284


In [18]:
# --- BLOCK 14: CREATE LOADERS (Train/Val Split) ---
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader as GeoDataLoader

# 1. Extract labels just for the split logic
y_all = [data.y.item() for data in graph_list]

# 2. Simple Split (80% Train, 20% Val)
# We use 'stratify=y_all' to ensure Class 2 doesn't disappear (Standard practice)
train_graphs, val_graphs = train_test_split(
    graph_list,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

# 3. Create GNN Loaders
# GeoDataLoader automatically batches graphs together
train_loader = GeoDataLoader(train_graphs, batch_size=32, shuffle=True)
val_loader   = GeoDataLoader(val_graphs, batch_size=32, shuffle=False)

print(f"Train Set: {len(train_graphs)} graphs")
print(f"Val Set:   {len(val_graphs)} graphs")

# Verify Class 2 presence
y_val = [g.y.item() for g in val_graphs]
print("Validation Class Distribution:", {k: y_val.count(k) for k in set(y_val)})

Train Set: 1827 graphs
Val Set:   457 graphs
Validation Class Distribution: {0: 137, 1: 228, 2: 92}


In [19]:
import numpy as np

def sparse_to_image(A, H=128, W=128):
    """
    Helper function to convert sparse matrix to a fixed grid.
    Used by matrix_to_graph to determine node connectivity.
    """
    A = A.tocoo()
    n_rows, n_cols = A.shape

    if A.nnz == 0:
        return np.zeros((H, W), dtype=np.float32)

    rows = A.row.astype(np.float64)
    cols = A.col.astype(np.float64)
    vals = A.data.astype(np.float64)

    # 1) Scale coordinates to [0, H-1] and [0, W-1]
    r_scaled = np.floor(rows * (H - 1) / max(n_rows - 1, 1)).astype(int)
    c_scaled = np.floor(cols * (W - 1) / max(n_cols - 1, 1)).astype(int)

    # 2) Normalize |vals| to [0,1]
    v_abs = np.abs(vals)
    v_min, v_max = v_abs.min(), v_abs.max()
    if v_max - v_min < 1e-12:
        v_norm = np.zeros_like(v_abs)
    else:
        v_norm = (v_abs - v_min) / (v_max - v_min)

    # 3) Accumulate into image
    img = np.zeros((H, W), dtype=np.float32)
    np.add.at(img, (r_scaled, c_scaled), v_norm)

    # 4) Normalize whole image to [0,1]
    max_val = img.max()
    if max_val > 0:
        img /= max_val

    return img

In [20]:
# --- BLOCK 14: TRAIN/VAL SPLIT (GNN Version) ---
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader as GeoDataLoader
import numpy as np

# 1. Extract labels to ensure balanced splitting
# (We need to look inside the graph objects to get the labels)
y_all = [data.y.item() for data in graph_list]

# 2. Split Data (80% Train, 20% Val)
# We use 'stratify=y_all' to guarantee Class 2 (Adaptive) is in both sets
train_graphs, val_graphs = train_test_split(
    graph_list,
    test_size=0.2,
    stratify=y_all,
    random_state=42
)

# 3. Create GNN Loaders
# GeoDataLoader is special: it batches multiple graphs into one giant graph for efficiency
train_loader = GeoDataLoader(train_graphs, batch_size=32, shuffle=True)
val_loader   = GeoDataLoader(val_graphs, batch_size=32, shuffle=False)

print(f"Train Set: {len(train_graphs)} graphs")
print(f"Val Set:   {len(val_graphs)} graphs")

# Verify that Class 2 (Adaptive) exists in Validation
y_val = [g.y.item() for g in val_graphs]
unique, counts = np.unique(y_val, return_counts=True)
print("Validation Class Distribution:", dict(zip(unique, counts)))

Train Set: 1827 graphs
Val Set:   457 graphs
Validation Class Distribution: {np.int64(0): np.int64(137), np.int64(1): np.int64(228), np.int64(2): np.int64(92)}


In [21]:
# --- BLOCK 15: GNN MODEL DEFINITION ---
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class MatrixGCN(torch.nn.Module):
    def __init__(self, num_node_features=1, num_classes=3, hidden_dim=64):
        super(MatrixGCN, self).__init__()

        # 1. Graph Convolutional Layers (Learn Structure)
        # GCNConv passes messages along edges (non-zero entries)
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim * 2)
        self.conv3 = GCNConv(hidden_dim * 2, hidden_dim * 2)

        # 2. Normalization (Stabilizes training)
        self.bn1 = torch.nn.BatchNorm1d(hidden_dim)
        self.bn2 = torch.nn.BatchNorm1d(hidden_dim * 2)
        self.bn3 = torch.nn.BatchNorm1d(hidden_dim * 2)

        # 3. Classifier (Graph-level prediction)
        self.lin1 = torch.nn.Linear(hidden_dim * 2, hidden_dim)
        self.lin2 = torch.nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        # A. Message Passing (Node Embeddings)
        # Layer 1
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        # Layer 2
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        # Layer 3
        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        # B. Global Pooling (Node Embeddings -> Graph Embedding)
        # Averages all node features into a single vector for the whole matrix
        x = global_mean_pool(x, batch)

        # C. Final Classification
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)

        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MatrixGCN(num_node_features=1, num_classes=3).to(device)
print(f"GNN Model Initialized on {device}")

GNN Model Initialized on cuda


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        alpha: Class weights (tensor). pass your 'class_weights_tensor' here.
        gamma: Focusing parameter. Higher gamma (e.g., 2.0) focuses more on hard examples.
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # 1. Calculate standard Cross Entropy Loss
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)

        # 2. Get the probabilities for the correct class (p_t)
        pt = torch.exp(-ce_loss)

        # 3. Calculate Focal Loss: (1 - pt)^gamma * CE_Loss
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [25]:
# --- BLOCK 17: WEIGHTS, OPTIMIZER & CRITERION ---
import torch
import numpy as np

# 1. Get class counts from the actual graph dataset
y_all = np.array([data.y.item() for data in graph_list])
classes = np.unique(y_all)
class_counts = np.array([(y_all == c).sum() for c in classes], dtype=np.float32)

print("Class counts (Graph Dataset):", dict(zip(classes, class_counts)))

# 2. Calculate Inverse Frequency Weights (Square root helps prevent extreme weights)
class_weights = 1.0 / np.sqrt(class_counts)
class_weights = class_weights / class_weights.sum() * len(classes)

print("Class weights:", dict(zip(classes, class_weights)))
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)

# 3. Initialize Criterion, Optimizer, and Scheduler
# We use the FocalLoss you defined in the previous block
criterion = FocalLoss(alpha=class_weights_t, gamma=1.0)

# Adam optimizer with a slightly higher LR for GNN stability
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

Class counts (Graph Dataset): {np.int64(0): np.float32(682.0), np.int64(1): np.float32(1140.0), np.int64(2): np.float32(462.0)}
Class weights: {np.int64(0): np.float32(1.0038652), np.int64(1): np.float32(0.7764529), np.int64(2): np.float32(1.219682)}


In [26]:
# --- BLOCK 18: TRAINING LOOP (GNN ADAPTED) ---
from tqdm import tqdm
import copy

def run_epoch(loader, model, optimizer=None, device=device):
    if optimizer is None:
        model.eval()
    else:
        model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for data in loader:
        # Move the whole graph batch to GPU
        data = data.to(device)

        # GNN Forward Pass: pass node features, edge list, and batch mapping
        logits = model(data.x, data.edge_index, data.batch)
        loss = criterion(logits, data.y)

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Update metrics (data.num_graphs tells us how many matrices are in this batch)
        total_loss += loss.item() * data.num_graphs
        preds = logits.argmax(dim=1)
        total_correct += (preds == data.y).sum().item()
        total_samples += data.num_graphs

    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples
    return avg_loss, acc

# --- Main Training Loop ---
EPOCHS = 80
patience = 12  # Slightly more patience for GNN convergence
best_val_loss = float('inf')
best_state_dict = copy.deepcopy(model.state_dict())
patience_counter = 0

print("Starting GNN Training...")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, model, optimizer)
    val_loss, val_acc     = run_epoch(val_loader, model, optimizer=None)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.3f} | "
        f"LR: {current_lr:.6f}"
    )

    # Early stopping logic
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state_dict = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

# Restore best weights
model.load_state_dict(best_state_dict)
torch.save(model.state_dict(), "best_precision_gnn.pth")
print("Best weights restored and saved to best_precision_gnn.pth")

Starting GNN Training...
Epoch 01 | Train Loss: 0.6650, Acc: 0.430 | Val Loss: 0.6619, Acc: 0.464 | LR: 0.001000
Epoch 02 | Train Loss: 0.6577, Acc: 0.426 | Val Loss: 0.6616, Acc: 0.495 | LR: 0.001000
Epoch 03 | Train Loss: 0.6572, Acc: 0.462 | Val Loss: 0.6490, Acc: 0.505 | LR: 0.001000
Epoch 04 | Train Loss: 0.6542, Acc: 0.428 | Val Loss: 0.6375, Acc: 0.514 | LR: 0.001000
Epoch 05 | Train Loss: 0.6529, Acc: 0.456 | Val Loss: 0.6387, Acc: 0.442 | LR: 0.001000
Epoch 06 | Train Loss: 0.6508, Acc: 0.446 | Val Loss: 0.6484, Acc: 0.394 | LR: 0.001000
Epoch 07 | Train Loss: 0.6565, Acc: 0.419 | Val Loss: 0.6473, Acc: 0.466 | LR: 0.001000
Epoch 08 | Train Loss: 0.6528, Acc: 0.464 | Val Loss: 0.6436, Acc: 0.383 | LR: 0.001000
Epoch 09 | Train Loss: 0.6533, Acc: 0.412 | Val Loss: 0.6363, Acc: 0.492 | LR: 0.001000
Epoch 10 | Train Loss: 0.6513, Acc: 0.467 | Val Loss: 0.6457, Acc: 0.470 | LR: 0.001000
Epoch 11 | Train Loss: 0.6525, Acc: 0.450 | Val Loss: 0.6409, Acc: 0.473 | LR: 0.001000
Epoch 1

In [27]:
# --- BLOCK 19: EVALUATION & CONFUSION MATRIX ---
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import torch

model.eval()
all_preds = []
all_trues = []

print("Running evaluation on validation set...")

with torch.no_grad():
    for data in val_loader:
        # Move GNN data batch to GPU
        data = data.to(device)

        # Forward pass with graph-specific inputs
        logits = model(data.x, data.edge_index, data.batch)
        preds = logits.argmax(dim=1)

        all_preds.append(preds.cpu().numpy())
        all_trues.append(data.y.cpu().numpy())

# Flatten lists into numpy arrays
y_true = np.concatenate(all_trues)
y_pred = np.concatenate(all_preds)

print("Evaluation Complete.")
print("True labels sample: ", y_true[:20])
print("Pred labels sample:", y_pred[:20])

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
print("\nConfusion Matrix (rows = true, cols = predicted):")
print(cm)

# Classification Report
target_names = ["Entry-wise (0)", "Row-wise (1)", "Adaptive (2)"]
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names, digits=3))

Running evaluation on validation set...
Evaluation Complete.
True labels sample:  [1 1 1 0 2 1 1 2 1 1 1 1 0 2 1 0 1 2 0 1]
Pred labels sample: [1 2 1 0 0 1 1 1 1 1 1 0 2 0 1 0 1 2 0 0]

Confusion Matrix (rows = true, cols = predicted):
[[ 65  62  10]
 [ 50 144  34]
 [ 17  49  26]]

Classification Report:
                precision    recall  f1-score   support

Entry-wise (0)      0.492     0.474     0.483       137
  Row-wise (1)      0.565     0.632     0.596       228
  Adaptive (2)      0.371     0.283     0.321        92

      accuracy                          0.514       457
     macro avg      0.476     0.463     0.467       457
  weighted avg      0.504     0.514     0.507       457

